<a href="https://colab.research.google.com/github/fcovarrubiasp-debug/CIENCIA-DE-DATO-/blob/main/2_S1_Notebook_Base_CDD_Fernando_Covarrubias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 1 — Obtención del dato
**Ciencia de Datos · Ingeniería de Ejecución en Informática — Advance UCSC**

| | |
|---|---|
| **Semana** | 1 |
| **Resultado de aprendizaje** | RA1 (foco) · RA3 (transversal) · RA5 (apertura) |
| **Duración estimada** | 1 h asincrónica + 1 h de trabajo autónomo |
| **Estudiante** | _(Fernando Covarrubias )_ |

---

## Antes de empezar

Sube al entorno los tres archivos que acompañan esta guía:

- `S1_Dataset_Guia_CDD.csv`
- `S1_Base_Ejemplo_CDD.sqlite`
- `S1_Respuesta_API_Ejemplo_CDD.json`

En Google Colab: panel izquierdo → icono de carpeta → *Subir*. En Jupyter local: déjalos en la misma carpeta que este cuaderno.

## Celda 0 · Pregunta de activación

Antes de escribir código, responde aquí para ti (no se entrega):

> **¿Qué dato de tu propio trabajo no usarías hoy para tomar una decisión importante, y por qué?**

_Escribe tu respuesta reemplazando esta línea._

## 1 · Preparación del entorno

Un flujo reproducible parte por declarar qué versiones se usaron. Si alguien ejecuta este cuaderno dentro de un año y obtiene otro resultado, lo primero que va a mirar es esto.

In [ ]:
import sys, platform
import pandas as pd
import numpy as np
import os

# Adaptamos la RUTA para que use los nombres reales de los archivos cargados en el entorno
# (detectados con prefijos numéricos en /content/)
if os.path.exists('/content/6-S1_Dataset_Guia_CDD.csv'):
    RUTA = '/content/6-'
else:
    RUTA = ''

SEMILLA = 20261
np.random.seed(SEMILLA)

print('Python    :', sys.version.split()[0], '·', platform.system())
print('pandas    :', pd.__version__)
print('numpy     :', np.__version__)
print('semilla   :', SEMILLA)
print('RUTA prefijo:', f'"{RUTA}"')

Python    : 3.13.16 · Linux
pandas    : 2.2.3
numpy     : 2.1.3
semilla   : 20261
RUTA prefijo: "/content/6-"


## 2 · Fuente 1 — Archivo plano (CSV)

El primer problema de un archivo plano nunca es el análisis: es **abrirlo bien**. Tres cosas pueden salir mal antes de ver un solo número: la codificación, el separador y los tipos de dato.

Ejecuta la celda siguiente y observa el resultado. Después responde: ¿cuál es el separador de este archivo?

In [ ]:
# Lectura deliberadamente ingenua: mira lo que pasa
try:
    crudo = pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv')
    print('Forma:', crudo.shape)
    display(crudo.head(3))
except Exception as e:
    print(type(e).__name__)
    print(e)

ParserError
Error tokenizing data. C error: Expected 1 fields in line 39, saw 2



**Lee el error, no lo saltes.** Dice algo como *«Expected 1 fields in line 39, saw 2»*.

Se traduce así: pandas asumió que el separador era la coma, no encontró ninguna en la primera línea y concluyó que el archivo tenía **una sola columna**. Todo iba bien hasta la línea 39, donde apareció una coma —un decimal escrito a la chilena, `12,4`— y esa fila pasó a tener dos campos. Contradicción, y se detiene.

Son dos defectos a la vez: el separador real es `;` y además hay decimales con coma. Resolvamos primero el separador.

In [ ]:
aire = pd.read_csv(
    RUTA + 'S1_Dataset_Guia_CDD.csv',
    sep=';',
    encoding='utf-8',
)
print('Forma:', aire.shape)
aire.head(40)

Forma: (734, 11)


,id_registro,estacion_id,estacion,comuna,region,fecha,pm25,pm10,temperatura_c,humedad_pct,validado
0,02-20260626,2,Los Ángeles Oriente,Los Ángeles,Biobío,2026-06-26,77.1,154.9,9.5,58.3,SI
1,08-20260405,8,Coronel Norte,Coronel,Biobío,2026-04-05,31.6,52.0,12.7,65.4,SI
2,06-20260621,6,Osorno Entre Lagos,Osorno,Los Lagos,2026-06-21,63.5,111.4,6.7,83.0,SI
3,01-20260531,1,Kingston,Concepción,Biobío,2026-05-31,48.1,79.4,14.4,75.1,SI
4,05-20260522,5,Temuco Padre Las Casas,Temuco,Araucanía,2026-05-22,85.0,162.1,11.5,82.9,SI
5,04-20260616,4,Talcahuano,Talcahuano,Biobío,2026-06-16,41.3,84.2,6.6,88.5,SI
6,06-20260501,6,Osorno Entre Lagos,Osorno,Los Lagos,2026-05-01,NaN,130.9,14.0,56.1,SI
7,08-20260531,8,Coronel Norte,Coronel,Biobío,2026-05-31,52.0,104.8,11.0,71.3,SI
8,02-20260512,2,Los Ángeles Oriente,Los Ángeles,Biobío,2026-05-12,77.6,136.3,14.3,89.9,SI
9,05-20260618,5,Temuco Padre Las Casas,Temuco,Araucanía,2026-06-18,63.9,110.2,9.3,93.7,SI


In [ ]:
# Tipos de dato con los que pandas interpretó cada columna
aire.dtypes

,0
id_registro,object
estacion_id,int64
estacion,object
comuna,object
region,object
fecha,object
pm25,float64
pm10,float64
temperatura_c,object
humedad_pct,float64


### ✏️ Actividad 2.1

Mira los tipos con atención. `pm25` y `pm10` quedaron numéricos, pero `temperatura_c` quedó como `object`, es decir, texto. Que una columna numérica quede como texto **siempre** significa que hay algo dentro que no es un número: aquí son los nueve decimales escritos con coma que provocaron el error anterior.

Y existe la trampa inversa: `pm10` es numérico y aun así está contaminado. Lo verás en la actividad 3.2.

**Escribe el código** que muestre los valores no numéricos de `temperatura_c`. Pista: `pd.to_numeric(..., errors='coerce')` convierte a `NaN` lo que no puede convertir.

In [ ]:
# Forzamos la conversión a numérico: lo que tenga comas se convertirá en NaN
temp_numerica = pd.to_numeric(aire['temperatura_c'], errors='coerce')

# Filtramos las filas donde la conversión falló pero el valor original no era nulo
valores_conflictivos = aire[temp_numerica.isna() & aire['temperatura_c'].notna()]

# Mostramos el resultado
print(f"Cantidad de registros con problemas de formato: {len(valores_conflictivos)}")
display(valores_conflictivos[['id_registro', 'estacion', 'temperatura_c']])

Cantidad de registros con problemas de formato: 9


,id_registro,estacion,temperatura_c
37,02-20260603,Los Ángeles Oriente,"7,8"
209,07-20260615,Valdivia Centro,"8,3"
347,08-20260517,Coronel Norte,"15,0"
429,03-20260514,Coyhanco,"10,5"
522,04-20260515,Talcahuano,"8,8"
563,07-20260518,Valdivia Centro,"13,3"
603,05-20260408,Temuco Padre Las Casas,"16,4"
618,04-20260426,Talcahuano,"10,1"
702,03-20260626,Coyhanco,"10,4"


In [ ]:
# Intentamos convertir 'temperatura_c' a numérico de forma forzada.
# Aquellos valores que tengan coma se transformarán en NaN (not a number).
temp_numerica = pd.to_numeric(aire['temperatura_c'], errors='coerce')

# Filtramos el DataFrame original para mostrar únicamente las filas
# donde la conversión falló (donde el resultado es NaN), pero que originalmente no estaban vacías.
valores_conflictivos = aire[temp_numerica.isna() & aire['temperatura_c'].notna()]

# Mostramos los valores originales problemáticos y cuántos son
print(f"Cantidad de registros con problemas de formato: {len(valores_conflictivos)}")
print("\nAlgunos ejemplos de valores no numéricos encontrados:")
display(valores_conflictivos[['id_registro', 'estacion', 'temperatura_c']].head(10))

Cantidad de registros con problemas de formato: 9

Algunos ejemplos de valores no numéricos encontrados:


,id_registro,estacion,temperatura_c
37,02-20260603,Los Ángeles Oriente,"7,8"
209,07-20260615,Valdivia Centro,"8,3"
347,08-20260517,Coronel Norte,"15,0"
429,03-20260514,Coyhanco,"10,5"
522,04-20260515,Talcahuano,"8,8"
563,07-20260518,Valdivia Centro,"13,3"
603,05-20260408,Temuco Padre Las Casas,"16,4"
618,04-20260426,Talcahuano,"10,1"
702,03-20260626,Coyhanco,"10,4"


### ✏️ Actividad 2.1

Mira los tipos con atención. `pm25` y `pm10` quedaron numéricos, pero `temperatura_c` quedó como `object`, es decir, texto. Que una columna numérica quede como texto **siempre** significa que hay algo dentro que no es un número: aquí son los nueve decimales escritos con coma que provocaron el error anterior.

Y existe la trampa inversa: `pm10` es numérico y aun así está contaminado. Lo verás en la actividad 3.2.

**Escribe el código** que muestre los valores no numéricos de `temperatura_c`. Pista: `pd.to_numeric(..., errors='coerce')` convierte a `NaN` lo que no puede convertir.

## 3 · Perfilado inicial: ¿qué tengo entre manos?

Antes de analizar, hay que **conocer el dato**. Estas cuatro celdas son el mínimo que un analista ejecuta siempre, en este orden.

In [ ]:
# a) Dimensiones y nombres de columna
print('Filas   :', aire.shape[0])
print('Columnas:', aire.shape[1])
print()
print(aire.columns.tolist())

Filas   : 734
Columnas: 11

['id_registro', 'estacion_id', 'estacion', 'comuna', 'region', 'fecha', 'pm25', 'pm10', 'temperatura_c', 'humedad_pct', 'validado']


In [ ]:
# b) Valores faltantes por columna
faltantes = aire.isna().sum().to_frame('faltantes')
faltantes['porcentaje'] = (faltantes['faltantes'] / len(aire) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

,faltantes,porcentaje
pm25,35,4.77
estacion_id,0,0.00
id_registro,0,0.00
estacion,0,0.00
comuna,0,0.00
region,0,0.00
fecha,0,0.00
pm10,0,0.00
temperatura_c,0,0.00
humedad_pct,0,0.00


In [ ]:
# c) Filas duplicadas completas
print('Duplicados exactos:', aire.duplicated().sum())
print('id_registro repetidos:', aire['id_registro'].duplicated().sum())

Duplicados exactos: 14
id_registro repetidos: 14


In [ ]:
# d) Cardinalidad de las columnas categóricas
for col in ['estacion', 'comuna', 'region', 'validado']:
    print(f'{col:15s} · {aire[col].nunique():3d} valores distintos')

estacion        ·   8 valores distintos
comuna          ·  16 valores distintos
region          ·   5 valores distintos
validado        ·   1 valores distintos


### ✏️ Actividad 3.1

La columna `comuna` declara más valores distintos de los que deberían existir. **Ejecuta** `aire['comuna'].unique()` y explica en la celda de texto siguiente por qué ocurre.

Ojo: no lo corrijas todavía. Esta semana **detectamos**; en la semana 2 **corregimos** y dejamos registrada la decisión en la bitácora.

In [ ]:
aire['comuna'].unique()

array(['Los Ángeles', 'Coronel', 'Osorno', 'Concepción', 'Temuco',
       'Talcahuano', 'Chillán', 'Valdivia', 'TEMUCO  ', 'VALDIVIA  ',
       'LOS ÁNGELES  ', 'TALCAHUANO  ', 'CHILLÁN  ', 'CONCEPCIÓN  ',
       'CORONEL  ', 'OSORNO  '], dtype=object)

inconsistencia y suciedad en los datos de texto (strings)


### ✏️ Actividad 3.2 — Valores centinela

Un **valor centinela** es un número real que en realidad significa «no hay dato»: `-999`, `0`, `9999`. Si no lo detectas, entra al promedio y lo arruina.

**Escribe el código** que cuente cuántas veces aparece `-999` en `pm10` y calcula el promedio de `pm10` con y sin esos valores. Anota la diferencia.

In [ ]:
# 1. Contar la cantidad de valores centinela (-999)
cantidad = (aire['pm10'] == -999).sum()
print(f"Cantidad de registros con -999 en pm10: {cantidad}")

# 2. Calcular el promedio incluyendo los -999
promedio_con = aire['pm10'].mean()
print(f"Promedio CON valores centinela: {promedio_con:.2f}")

# 3. Calcular el promedio excluyendo los -999
# Reemplazamos temporalmente los -999 por NaN para que pandas los ignore en el promedio
pm10_limpio = aire['pm10'].replace(-999, np.nan)
promedio_sin = pm10_limpio.mean()
print(f"Promedio SIN valores centinela: {promedio_sin:.2f}")

# 4. Diferencia
diferencia = promedio_sin - promedio_con
print(f"Diferencia (impacto del valor centinela): {diferencia:.2f}")

Cantidad de registros con -999 en pm10: 18
Promedio CON valores centinela: 77.10
Promedio SIN valores centinela: 104.15
Diferencia (impacto del valor centinela): 27.05


## 4 · Fuente 2 — Base de datos relacional (SQL)

Buena parte de los datos de una organización no está en archivos, sino en una base de datos. El archivo `S1_Base_Ejemplo_CDD.sqlite` contiene tres tablas: `estacion`, `medicion` y `episodio`.

SQLite viene incluido en Python: no hay que instalar nada.

In [ ]:
import sqlite3
import os

# Ajustamos el prefijo de ruta para sqlite si es necesario
sqlite_path = RUTA + 'S1_Base_Ejemplo_CDD.sqlite'
if not os.path.exists(sqlite_path) and os.path.exists('/content/7-S1_Base_Ejemplo_CDD.sqlite'):
    sqlite_path = '/content/7-S1_Base_Ejemplo_CDD.sqlite'

con = sqlite3.connect(sqlite_path)

# ¿Qué tablas hay?
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", con)

,name
0,estacion
1,medicion
2,sqlite_sequence
3,episodio


In [ ]:
# Estructura de una tabla
pd.read_sql_query('PRAGMA table_info(episodio)', con)

,cid,name,type,notnull,dflt_value,pk
0,0,episodio_id,INTEGER,0,None,1
1,1,estacion_id,INTEGER,1,None,0
2,2,fecha,TEXT,1,None,0
3,3,categoria,TEXT,1,None,0


In [ ]:
# Una consulta con JOIN y agregación: promedio de PM2.5 por comuna
consulta = '''
SELECT  e.comuna,
        e.region,
        COUNT(*)              AS n_mediciones,
        ROUND(AVG(m.pm25), 1) AS pm25_promedio,
        ROUND(MAX(m.pm25), 1) AS pm25_maximo
FROM    medicion m
JOIN    estacion e ON e.estacion_id = m.estacion_id
GROUP BY e.comuna, e.region
ORDER BY pm25_promedio DESC
'''
pd.read_sql_query(consulta, con)

,comuna,region,n_mediciones,pm25_promedio,pm25_maximo
0,Temuco,Araucanía,90,74.2,102.1
1,Chillán,Ñuble,90,65.8,98.5
2,Los Ángeles,Biobío,90,62.2,87.5
3,Osorno,Los Lagos,90,60.3,82.5
4,Valdivia,Los Ríos,90,53.9,78.0
5,Coronel,Biobío,90,49.0,76.5
6,Concepción,Biobío,90,45.8,70.7
7,Talcahuano,Biobío,90,41.3,67.8


### ✏️ Actividad 4.1

**Escribe una consulta SQL** que responda: *¿cuántos episodios de cada categoría (`Alerta`, `Preemergencia`, `Emergencia`) registró cada comuna?*

Necesitas unir `episodio` con `estacion` y agrupar por dos columnas.

In [ ]:
# Aseguramos que la conexión esté abierta antes de consultar
import sqlite3

sqlite_path = RUTA + 'S1_Base_Ejemplo_CDD.sqlite'
if not os.path.exists(sqlite_path) and os.path.exists('/content/7-S1_Base_Ejemplo_CDD.sqlite'):
    sqlite_path = '/content/7-S1_Base_Ejemplo_CDD.sqlite'

con = sqlite3.connect(sqlite_path)

consulta_2 = '''
SELECT  e.comuna,
        ep.categoria,
        COUNT(*) AS total_episodios
FROM    episodio ep
JOIN    estacion e ON e.estacion_id = ep.estacion_id
GROUP BY e.comuna, ep.categoria
ORDER BY e.comuna, total_episodios DESC
'''

# Ejecutamos la consulta y mostramos los resultados
resultados_episodios = pd.read_sql_query(consulta_2, con)
display(resultados_episodios)

,comuna,categoria,total_episodios
0,Chillán,Alerta,5
1,Chillán,Preemergencia,1
2,Los Ángeles,Alerta,6
3,Osorno,Alerta,5
4,Temuco,Alerta,25
5,Temuco,Preemergencia,4


In [ ]:
con.close()

## 5 · Fuente 3 — Servicio web (API) y JSON anidado

Una API devuelve **JSON**, no una tabla. El JSON es jerárquico: dentro de un registro puede haber otro objeto, y dentro de ese, una lista. Convertirlo en una tabla rectangular es parte del trabajo.

Usamos aquí la respuesta guardada en archivo. En un caso real la obtendrías con `requests.get(url).json()`; el archivo existe para que el laboratorio funcione aunque el servicio esté caído.

In [ ]:
import json
import os

# Ajustamos el prefijo de ruta para el JSON si es necesario
json_path = RUTA + 'S1_Respuesta_API_Ejemplo_CDD.json'
if not os.path.exists(json_path) and os.path.exists('/content/8-S1_Respuesta_API_Ejemplo_CDD.json'):
    json_path = '/content/8-S1_Respuesta_API_Ejemplo_CDD.json'

with open(json_path, encoding='utf-8') as fh:
    respuesta = json.load(fh)

print('Claves de primer nivel:', list(respuesta.keys()))
print()
print('Metadatos:')
for k, v in respuesta['meta'].items():
    print(f'  {k}: {v}')

Claves de primer nivel: ['meta', 'data']

Metadatos:
  servicio: API de ejemplo · Red de monitoreo de calidad del aire (simulada para docencia)
  version: 1.2
  generado: 2026-09-21T10:00:00-03:00
  licencia: CC BY 4.0 (conjunto sintético de uso docente)
  paginacion: {'pagina': 1, 'por_pagina': 100, 'total': 100, 'siguiente': None}


In [ ]:
# Un registro cualquiera, para entender la estructura
print(json.dumps(respuesta['data'][0], ensure_ascii=False, indent=2))

{
  "codigo_estacion": "EST-001",
  "ubicacion": {
    "comuna": "Concepción",
    "region": "Biobío",
    "coordenadas": {
      "lat": -36.827,
      "lon": -73.0498
    }
  },
  "fecha": "2026-04-01",
  "mediciones": [
    {
      "parametro": "pm25",
      "valor": 37.6,
      "unidad": "ug/m3"
    },
    {
      "parametro": "pm10",
      "valor": 58.8,
      "unidad": "ug/m3"
    },
    {
      "parametro": "temperatura",
      "valor": 13.1,
      "unidad": "C"
    }
  ],
  "calidad_dato": "preliminar"
}


In [ ]:
# json_normalize aplana la jerarquía: fíjate en los nombres de columna que genera
api = pd.json_normalize(respuesta['data'])
api.head(3)

,codigo_estacion,fecha,mediciones,calidad_dato,ubicacion.comuna,ubicacion.region,ubicacion.coordenadas.lat,ubicacion.coordenadas.lon
0,EST-001,2026-04-01,"[{'parametro': 'pm25', 'valor': 37.6, 'unidad'...",preliminar,Concepción,Biobío,-36.827,-73.0498
1,EST-001,2026-04-02,"[{'parametro': 'pm25', 'valor': 25.6, 'unidad'...",validado,Concepción,Biobío,-36.827,-73.0498
2,EST-001,2026-04-03,"[{'parametro': 'pm25', 'valor': 29.3, 'unidad'...",validado,Concepción,Biobío,-36.827,-73.0498


In [ ]:
# La lista 'mediciones' sigue anidada. record_path la despliega en filas:
api_largo = pd.json_normalize(
    respuesta['data'],
    record_path='mediciones',
    meta=['codigo_estacion', 'fecha', ['ubicacion', 'comuna']],
)
api_largo.head(6)

,parametro,valor,unidad,codigo_estacion,fecha,ubicacion.comuna
0,pm25,37.6,ug/m3,EST-001,2026-04-01,Concepción
1,pm10,58.8,ug/m3,EST-001,2026-04-01,Concepción
2,temperatura,13.1,C,EST-001,2026-04-01,Concepción
3,pm25,25.6,ug/m3,EST-001,2026-04-02,Concepción
4,pm10,63.5,ug/m3,EST-001,2026-04-02,Concepción
5,temperatura,15.6,C,EST-001,2026-04-02,Concepción


### ✏️ Actividad 5.1

El resultado anterior está en **formato largo**: una fila por parámetro medido. Para analizar conviene el **formato ancho**: una fila por estación-fecha y una columna por parámetro.

**Escribe el código** que haga esa transformación con `pivot_table`.

In [ ]:
# Transformamos el DataFrame de formato largo a formato ancho
api_ancho = api_largo.pivot_table(
    index=['codigo_estacion', 'fecha', 'ubicacion.comuna'],
    columns='parametro',
    values='valor'
).reset_index()

# Mostramos el resultado ordenado y limpio
display(api_ancho.head(10))

parametro,codigo_estacion,fecha,ubicacion.comuna,pm10,pm25,temperatura
0,EST-001,2026-04-01,Concepción,58.8,37.6,13.1
1,EST-001,2026-04-02,Concepción,63.5,25.6,15.6
2,EST-001,2026-04-03,Concepción,75.4,29.3,13.3
3,EST-001,2026-04-04,Concepción,55.9,33.0,11.7
4,EST-001,2026-04-05,Concepción,79.1,39.1,12.4
5,EST-001,2026-04-06,Concepción,76.4,42.9,16.3
6,EST-001,2026-04-07,Concepción,69.8,33.7,11.1
7,EST-001,2026-04-08,Concepción,61.3,30.7,11.0
8,EST-001,2026-04-09,Concepción,81.4,29.9,10.8
9,EST-001,2026-04-10,Concepción,75.4,37.0,16.8


## 6 · Fuente 4 — Extracción web

Cuando el dato solo existe publicado en una página, hay que extraerlo. `pandas.read_html` resuelve el caso más común: una tabla HTML.

**Antes de extraer, tres preguntas que no son técnicas:** ¿los términos de uso del sitio lo permiten?, ¿estás generando una carga razonable?, ¿el dato incluye personas? Si alguna respuesta te incomoda, no extraigas.

In [ ]:
from io import StringIO
import requests
import pandas as pd
# Requiere conexión a internet. Si no la tienes, salta esta celda.
URL = 'https://es.wikipedia.org/wiki/Anexo:Comunas_de_Chile'   # [DOCENTE: confirmar o reemplazar]

try:
    headers = {'User-Agent': 'Mozilla/5.0'}
    respuesta = requests.get(URL, headers=headers)
    respuesta.raise_for_status()
    tablas = pd.read_html(StringIO(respuesta.text))
    print('Tablas encontradas:', len(tablas))
    web = tablas[0]
    display(web.head())
except Exception as e:
    print('No se pudo extraer:', type(e).__name__, e)
    print('Continúa con el resto del laboratorio.')

Tablas encontradas: 2


,CUT (Código Único Territorial),Nombre,Unnamed: 2,Provincia,Región,Superficie (km²),Población 2024 (Censo),Densidad (hab./km²),IDH 2005,IDH 2005.1,Latitud,Longitud
0,15101,Arica,NaN,Arica,Arica y Parinacota,"4.799,4",241 653,51.6,0.866,Muy alto,"-18°27'18""","-70°17'24"""
1,15102,Camarones,NaN,Arica,Arica y Parinacota,3.927,861,0.31,0.791,Alto,"-19°1'1.2""","-69°52'1.2"""
2,15201,Putre,NaN,Parinacota,Arica y Parinacota,"5.902,5",1 547,0.43,0.817,Muy alto,"-18°12'0""","-69°34'58.8"""
3,15202,General Lagos,NaN,Parinacota,Arica y Parinacota,"2.244,4",508,0.36,0.773,Alto,"-17°39'10.8""","-69°38'6"""
4,1101,Iquique,NaN,Iquique,Tarapacá,"2.242,1",199 587,996,0.826,Muy alto,"-20°14'38.4""","-70°8'20.4"""


### ✏️ Actividad 6.1

Responde en la celda de texto siguiente: **¿qué diferencia hay, para tu proyecto, entre un dato descargado de un portal oficial y uno extraído de una página web?** Piensa en trazabilidad y en qué pasa si mañana la página cambia.

La diferencia es que los datos son verificables , analizados , que no se pueden modificar son verificable y reproducible, una pagina confiable puede ser quizas ventajosa pero no validada que puede causar errores



## 7 · Volumen: cuando el conjunto no cabe

No todo conjunto se lee entero. Cuando el archivo supera la memoria disponible hay tres salidas: **leer solo las columnas necesarias**, **leer por bloques** o **trabajar con una muestra**.

Una muestra bien construida responde muchas preguntas con una fracción del costo. Una muestra mal construida responde mal todas.

In [ ]:
# Leer solo algunas columnas: menos memoria desde el inicio
liviano = pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv', sep=';',
                      usecols=['estacion', 'fecha', 'pm25'])
print('Memoria completo :', aire.memory_usage(deep=True).sum() / 1024, 'KB')
print('Memoria liviano  :', liviano.memory_usage(deep=True).sum() / 1024, 'KB')

Memoria completo : 335.0009765625 KB
Memoria liviano  : 94.8486328125 KB


In [ ]:
# Lectura por bloques: se procesa de a trozos y nunca se carga todo
total, filas = 0, 0
for bloque in pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv', sep=';', chunksize=100):
    valores = pd.to_numeric(bloque['pm25'], errors='coerce').dropna()
    total += valores.sum()
    filas += len(valores)
print(f'Promedio de pm25 calculado por bloques: {total / filas:.2f}')

Promedio de pm25 calculado por bloques: 62.04


In [ ]:
# Muestreo aleatorio simple vs. estratificado por estación
simple = aire.sample(n=120, random_state=SEMILLA)
estratificado = (aire.groupby('estacion', group_keys=False)
                     .apply(lambda g: g.sample(n=15, random_state=SEMILLA)))

print('Distribución en la muestra SIMPLE:')
print(simple['estacion'].value_counts().to_string())
print()
print('Distribución en la muestra ESTRATIFICADA:')
print(estratificado['estacion'].value_counts().to_string())

Distribución en la muestra SIMPLE:
estacion
Kingston                  18
Osorno Entre Lagos        18
Coyhanco                  17
Los Ángeles Oriente       17
Coronel Norte             16
Talcahuano                13
Temuco Padre Las Casas    13
Valdivia Centro            8

Distribución en la muestra ESTRATIFICADA:
estacion
Coronel Norte             15
Coyhanco                  15
Kingston                  15
Los Ángeles Oriente       15
Osorno Entre Lagos        15
Talcahuano                15
Temuco Padre Las Casas    15
Valdivia Centro           15


/tmp/ipykernel_2676/748870216.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=15, random_state=SEMILLA)))


### ✏️ Actividad 7.1

Mira las dos distribuciones anteriores. **¿En qué situación la muestra simple te llevaría a una conclusión equivocada?** Escríbelo abajo con un ejemplo concreto de tu propio conjunto de datos.

_el muestreo estrateficado es coveniente ya aue evita mayor representacion que otra sus resultados son mas equilibrados en sus numeros , es una representacion pareja al ser todos 15 son mas validos



## 8 · Tu conjunto de datos: la elección del proyecto

Abre `S1_Catalogo_Datasets_Sugeridos_CDD.xlsx`, revisa **al menos tres** alternativas y elige una. Registra tu elección en la hoja «Mi elección» del mismo archivo.

Luego descárgalo, cárgalo aquí abajo y ejecuta sobre él el mismo perfilado de la sección 3.

In [ ]:
import pandas as pd
import os

nombre_archivo = 'S1_Catalogo_Datasets_Sugeridos_CDD.xlsx'
mi_ruta_dataset = '/content/9-' + nombre_archivo if os.path.exists('/content/9-' + nombre_archivo) else nombre_archivo

try:
    # Cargamos el archivo omitiendo las filas de instrucciones del título (header en la fila índice 3)
    mi_dataset = pd.read_excel(mi_ruta_dataset, sheet_name='Catálogo', header=3)
    print(f'¡Catálogo de Datasets Sugeridos "{nombre_archivo}" cargado de manera correcta!')
    print('Dimensiones reales del catálogo:', mi_dataset.shape)
    display(mi_dataset.head())
except Exception as e:
    print('Error al cargar el catálogo de Excel. Asegúrate de que el archivo esté subido en /content/:')
    print(type(e).__name__, e)

¡Catálogo de Datasets Sugeridos "S1_Catalogo_Datasets_Sugeridos_CDD.xlsx" cargado de manera correcta!
Dimensiones reales del catálogo: (16, 10)


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,N°,Nombre del conjunto de datos,Portal / fuente,Tema,Formato,Tamaño aprox.,¿Datos personales?,Dificultad,Pregunta analítica sugerida,Enlace (verificar antes del montaje)
1,1,Calidad del aire — estaciones de monitoreo (SI...,Ministerio del Medio Ambiente / datos.gob.cl,Medio ambiente,CSV,> 100.000 filas,No,Media,¿Qué estaciones concentran los episodios críti...,https://datos.gob.cl — buscar «calidad del aire»
2,2,Estadísticas de accidentes de tránsito,CONASET / datos.gob.cl,Transporte y seguridad,CSV / XLSX,50.000–200.000 filas,No (agregados),Media,¿Qué factores se asocian a los tramos horarios...,https://datos.gob.cl — buscar «accidentes de t...
3,3,Encuesta Nacional de Empleo (ENE),Instituto Nacional de Estadísticas,Empleo,CSV,> 300.000 filas,No (anonimizada),Alta,¿Cómo varía la tasa de ocupación por región y ...,https://www.ine.gob.cl — sección Estadísticas ...
4,4,Índices de precios al consumidor (IPC),Instituto Nacional de Estadísticas,Economía,CSV / XLSX,Serie mensual larga,No,Baja,¿Qué divisiones del IPC explican la variación ...,https://www.ine.gob.cl — sección IPC


In [ ]:
# a) Dimensiones y nombres de columna
print('--- DIMENSIONES Y COLUMNAS ---')
print('Filas   :', mi_dataset.shape[0])
print('Columnas:', mi_dataset.shape[1])
print('Nombres de columnas:', mi_dataset.columns.tolist())
print('\n' + '='*50 + '\n')

# b) Valores faltantes por columna
print('--- VALORES FALTANTES ---')
faltantes_mi = mi_dataset.isna().sum().to_frame('faltantes')
faltantes_mi['porcentaje'] = (faltantes_mi['faltantes'] / len(mi_dataset) * 100).round(2)
display(faltantes_mi.sort_values('faltantes', ascending=False))
print('\n' + '='*50 + '\n')

# c) Filas duplicadas completas
print('--- DUPLICADOS ---')
print('Duplicados exactos completos:', mi_dataset.duplicated().sum())
if 'id_registro' in mi_dataset.columns:
    print('id_registro repetidos:', mi_dataset['id_registro'].duplicated().sum())
print('\n' + '='*50 + '\n')

# d) Cardinalidad de las columnas categóricas típicas
print('--- CARDINALIDAD DE CATEGÓRICAS ---')
columnas_categoricas = mi_dataset.select_dtypes(include=['object']).columns
for col in columnas_categoricas:
    print(f'{col:15s} · {mi_dataset[col].nunique():3d} valores distintos')

--- DIMENSIONES Y COLUMNAS ---
Filas   : 19
Columnas: 10
Nombres de columnas: ['Catálogo de conjuntos de datos sugeridos para el Proyecto Analítico', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']


--- VALORES FALTANTES ---


,faltantes,porcentaje
Unnamed: 1,6,31.58
Unnamed: 2,6,31.58
Unnamed: 6,6,31.58
Unnamed: 3,6,31.58
Unnamed: 4,6,31.58
Unnamed: 5,6,31.58
Unnamed: 8,6,31.58
Unnamed: 7,6,31.58
Unnamed: 9,6,31.58
Catálogo de conjuntos de datos sugeridos para el Proyecto Analítico,2,10.53




--- DUPLICADOS ---
Duplicados exactos completos: 1


--- CARDINALIDAD DE CATEGÓRICAS ---
Catálogo de conjuntos de datos sugeridos para el Proyecto Analítico ·  17 valores distintos
Unnamed: 1      ·  13 valores distintos
Unnamed: 2      ·  10 valores distintos
Unnamed: 3      ·  12 valores distintos
Unnamed: 4      ·   5 valores distintos
Unnamed: 5      ·  11 valores distintos
Unnamed: 6      ·   8 valores distintos
Unnamed: 7      ·   4 valores distintos
Unnamed: 8      ·  13 valores distintos
Unnamed: 9      ·  13 valores distintos


### ✏️ Actividad 8.1 — Pregunta analítica y unidad de análisis

Completa estas dos líneas. Son la base de todo el curso y las vas a necesitar en la Entrega 1.

- **Mi pregunta analítica es:** ¿Qué impacto tiene el índice de vulnerabilidad escolar (IVE) en el rendimiento académico promedio de los estudiantes, controlado por el tipo de dependencia del establecimiento?
- **Una fila de mi conjunto representa:** Una fila representa el registro de rendimiento y matrícula de un estudiante específico en un establecimiento y año determinado.
## ***Criterio para saber si la pregunta está bien formulada: si al leerla puedes nombrar qué columnas necesitas para responderla, está bien. Si no, es demasiado vaga.***

## 9 · Riesgo ético y legal

Antes de usar tu conjunto, completa la ficha `S1_Ficha_Riesgo_Etico_CDD.docx`. No es un trámite: si tu conjunto contiene datos de personas, lo que decidas ahí condiciona todo lo que puedes hacer en las semanas siguientes.

Esa ficha es el insumo de tu intervención en el **foro de debate** de esta semana y se cierra en la Sección C de la Entrega Final.

## 10 · Repositorio y control de versiones

Tu trabajo tiene que poder reconstruirse. Sube este cuaderno a un repositorio y deja el enlace aquí.

Desde Colab: *Archivo → Guardar una copia en GitHub*. Desde local:

In [ ]:
# Ejecuta esto en una terminal, no en el cuaderno:
# git init
# git add S1_Notebook_Base_CDD.ipynb
# git commit -m 'Laboratorio 1: obtención del dato'
# git remote add origin https://github.com/USUARIO/proyecto-ciencia-datos.git
# git push -u origin main


**Enlace a mi repositorio:** _…_

---

## Cierre del laboratorio

Al terminar deberías tener:

- [ ] El entorno funcionando y las versiones declaradas
- [ ] Datos leídos desde las cuatro fuentes
- [ ] El perfilado inicial del conjunto guía, con sus defectos detectados (no corregidos)
- [ ] Tu conjunto de datos elegido y registrado en el catálogo
- [ ] Tu pregunta analítica y tu unidad de análisis escritas
- [ ] La ficha de riesgo ético-legal completada
- [ ] El repositorio creado y este cuaderno versionado

Lo que quede pendiente, llévalo a la sesión sincrónica.

_Ciencia de Datos · Advance UCSC · Semana 1_